In [1]:
import fsspec
import pandas as pd

S3_REGION = "ap-northeast-1"
SO = {"client_kwargs": {"region_name": S3_REGION}}

SYMBOL = "BTCUSDT"
YEAR = 2024
MONTHS = [1, 2, 3, 4, 5, 6]  # <-- 6 mois

fs = fsspec.filesystem("s3", **SO)

def list_month_parts(symbol: str, year: int, month: int) -> list[str]:
    part_glob = (
        f"s3://tradebot-config-tokyo/data/stageA/"
        f"symbol={symbol}/year={year}/month={month:02d}/part-*.parquet"
    )
    _, _, paths = fsspec.get_fs_token_paths(part_glob, storage_options=SO)
    paths = sorted(paths)

    # IMPORTANT: remettre s3:// devant si absent
    def to_s3_url(p: str) -> str:
        return p if p.startswith("s3://") else f"s3://{p}"

    return [to_s3_url(p) for p in paths]

all_paths = []
for m in MONTHS:
    p = list_month_parts(SYMBOL, YEAR, m)
    print(f"month={m:02d} -> n_parts={len(p)}  example={p[:1]}")
    all_paths.extend(p)

all_paths = sorted(all_paths)

print("\nTOTAL n_parts:", len(all_paths))
print("example:", all_paths[:3])

month=01 -> n_parts=5  example=['s3://tradebot-config-tokyo/data/stageA/symbol=BTCUSDT/year=2024/month=01/part-00000.parquet']
month=02 -> n_parts=5  example=['s3://tradebot-config-tokyo/data/stageA/symbol=BTCUSDT/year=2024/month=02/part-00000.parquet']
month=03 -> n_parts=5  example=['s3://tradebot-config-tokyo/data/stageA/symbol=BTCUSDT/year=2024/month=03/part-00000.parquet']
month=04 -> n_parts=5  example=['s3://tradebot-config-tokyo/data/stageA/symbol=BTCUSDT/year=2024/month=04/part-00000.parquet']
month=05 -> n_parts=5  example=['s3://tradebot-config-tokyo/data/stageA/symbol=BTCUSDT/year=2024/month=05/part-00000.parquet']
month=06 -> n_parts=5  example=['s3://tradebot-config-tokyo/data/stageA/symbol=BTCUSDT/year=2024/month=06/part-00000.parquet']

TOTAL n_parts: 30
example: ['s3://tradebot-config-tokyo/data/stageA/symbol=BTCUSDT/year=2024/month=01/part-00000.parquet', 's3://tradebot-config-tokyo/data/stageA/symbol=BTCUSDT/year=2024/month=01/part-00001.parquet', 's3://tradebot-conf

In [3]:
MIN_COLS = [
    "id_t",
    "label_A",
    "label_A_exit_reason",
    "label_A_pnl_net_bps",
    "label_A_exit_t_sec",
    "audit_spread_bps_entry",
    "audit_cost_bps",
    "audit_R_bps",
    "audit_tp_bps",
    "audit_sl_bps",
    "audit_rr_min",
    "audit_cost_R",
    "audit_p_thr_ev0",
    "cfg_horizon_sec",
    "cfg_decision_step_sec",
    "cfg_k_confirm_sec",
]

def read_parts_stream(paths, columns):
    for p in paths:
        yield pd.read_parquet(
            p,
            storage_options=SO,
            columns=columns,
            engine="pyarrow",
        )

dfs = []
n_rows = 0

for df_part in read_parts_stream(all_paths, MIN_COLS):
    n_rows += len(df_part)
    dfs.append(df_part)

df = pd.concat(dfs, ignore_index=True)

print("rows:", n_rows)
print("shape:", df.shape)
df.head()

rows: 506879
shape: (506879, 16)


,id_t,label_A,label_A_exit_reason,label_A_pnl_net_bps,label_A_exit_t_sec,audit_spread_bps_entry,audit_cost_bps,audit_R_bps,audit_tp_bps,audit_sl_bps,audit_rr_min,audit_cost_R,audit_p_thr_ev0,cfg_horizon_sec,cfg_decision_step_sec,cfg_k_confirm_sec
0,2024-01-01 00:00:00+00:00,0,SL_SHORT,-16.925575,55,0.023641,6.011820,10.913755,28.375765,10.913755,2.6,0.550848,0.430791,120,30,20
1,2024-01-01 00:00:30+00:00,0,SL_SHORT,-16.925570,82,0.023629,6.011815,10.913755,28.375765,10.913755,2.6,0.550847,0.430791,120,30,20
2,2024-01-01 00:01:00+00:00,0,NOFILL,0.000000,-1,0.023616,6.011808,10.489022,27.271458,10.489022,2.6,0.573152,0.436987,120,30,20
3,2024-01-01 00:01:30+00:00,0,SL_SHORT,-16.654305,107,0.330565,6.165283,10.489022,27.271458,10.489022,2.6,0.587784,0.441051,120,30,20
4,2024-01-01 00:02:00+00:00,0,SL_SHORT,-16.106838,76,0.023613,6.011806,10.095032,26.247082,10.095032,2.6,0.595521,0.443200,120,30,20


In [4]:
# --- 3)  
import numpy as np
import pandas as pd

print("shape:", df.shape)

# --- dtypes attendus (souple, mais utile)
EXPECTED_DTYPES = {
    "id_t": "datetime64[ns, UTC]",
    "label_A": "int8",
    "label_A_exit_t_sec": "int16",
    "audit_cost_bps": "float32",
    "audit_R_bps": "float32",
    "audit_tp_bps": "float32",
    "audit_sl_bps": "float32",
    "audit_cost_R": "float32",
    "audit_p_thr_ev0": "float32",
}

print("\n--- dtypes ---")
print(df.dtypes)

# --- checks simples
print("\n--- duplicates id_t ---")
dup = df["id_t"].duplicated().mean()
print("duplicate ratio:", dup)

print("\n--- monotonicity (si trié) ---")
df_sorted = df.sort_values("id_t")
mono = df_sorted["id_t"].is_monotonic_increasing
print("id_t monotonic increasing after sort:", mono)

print("\n--- NA ratios ---")
na = df.isna().mean().sort_values(ascending=False)
print(na.head(10))

# --- ranges sanity
print("\n--- ranges sanity ---")
print("label_A unique:", sorted(df["label_A"].dropna().unique().tolist()))
print("exit_t_sec min/max:", int(df["label_A_exit_t_sec"].min()), int(df["label_A_exit_t_sec"].max()))
print("p_thr_ev0 min/max:", float(df["audit_p_thr_ev0"].min()), float(df["audit_p_thr_ev0"].max()))

# hard asserts (si tu veux que ça fail vite)
assert df["label_A"].isin([0,1]).all()
assert df["audit_p_thr_ev0"].between(0.0, 1.0).all()
assert df["label_A_exit_t_sec"].between(-1, 120).all()

shape: (506879, 16)

--- dtypes ---
id_t                      datetime64[ns, UTC]
label_A                                  int8
label_A_exit_reason            string[python]
label_A_pnl_net_bps                   float32
label_A_exit_t_sec                      int16
audit_spread_bps_entry                float32
audit_cost_bps                        float32
audit_R_bps                           float32
audit_tp_bps                          float32
audit_sl_bps                          float32
audit_rr_min                          float32
audit_cost_R                          float32
audit_p_thr_ev0                       float32
cfg_horizon_sec                         int16
cfg_decision_step_sec                   int16
cfg_k_confirm_sec                       int16
dtype: object

--- duplicates id_t ---
duplicate ratio: 0.0

--- monotonicity (si trié) ---
id_t monotonic increasing after sort: True

--- NA ratios ---
id_t                      0.0
label_A                   0.0
label_A_exit_r

In [9]:
# --- 5) df["month"] = df["id_t"].dt.to_period("M").astype(str)

# global
print("N rows:", len(df))
print("label_A mean:", df["label_A"].mean())

er = df["label_A_exit_reason"].fillna("NA")
dist = er.value_counts()
dist_pct = (dist / len(df) * 100).round(3)
out = pd.DataFrame({"count": dist, "pct": dist_pct})
print("\nexit_reason distribution:\n", out)

# par mois
g = df.groupby("month").agg(
    n=("label_A","size"),
    label_mean=("label_A","mean"),
    tp=("label_A", "sum"),
    nofill=("label_A_exit_reason", lambda s: (s=="NOFILL").sum()),
    time=("label_A_exit_reason", lambda s: (s=="TIME").sum()),
    cf=("label_A_exit_reason", lambda s: (s=="CONFIRM_FAIL").sum()),
)
g["tp_rate"] = g["tp"]/g["n"]
g["nofill_rate"] = g["nofill"]/g["n"]
g["time_rate"] = g["time"]/g["n"]
g["cf_rate"] = g["cf"]/g["n"]

print("\n--- monthly summary ---")
display(g.sort_index())

N rows: 506879
label_A mean: 0.0036537319557527538

exit_reason distribution:
                       count     pct
label_A_exit_reason                
NOFILL               277205  54.689
TIME                 137803  27.187
CONFIRM_FAIL          43454   8.573
SL_SHORT              23645   4.665
SL_LONG               22917   4.521
TP_SHORT                974   0.192
TP_LONG                 878   0.173
NONE                      3   0.001

--- monthly summary ---


,n,label_mean,tp,nofill,time,cf,tp_rate,nofill_rate,time_rate,cf_rate
month,,,,,,,,,,
2024-01,89280,0.003371,301,46202,27164,6642,0.003371,0.517496,0.304256,0.074395
2024-02,80640,0.002269,183,50431,18385,6009,0.002269,0.625384,0.227989,0.074516
2024-03,83519,0.005879,491,39100,27401,6257,0.005879,0.468157,0.328081,0.074917
2024-04,83520,0.005795,484,38957,27363,6833,0.005795,0.466439,0.327622,0.081813
2024-05,86400,0.002975,257,47567,22326,8978,0.002975,0.550544,0.258403,0.103912
2024-06,83520,0.001628,136,54948,15164,8735,0.001628,0.657902,0.181561,0.104586


In [ ]:
# --- 4) 
er = df["label_A_exit_reason"].fillna("NA")

filled_loose = ~er.isin(["NOFILL"])
filled_strict = ~er.isin(["NOFILL","CONFIRM_FAIL"])

print("filled share (loose, not NOFILL):", float(filled_loose.mean()))
print("filled share (strict, not NOFILL/CONFIRM_FAIL):", float(filled_strict.mean()))

df_f = df.loc[filled_loose].copy()
dist_f = df_f["label_A_exit_reason"].value_counts(normalize=True).sort_values(ascending=False)
print("\nexit_reason (filled only):")
print(dist_f)

# check logique: TP => label_A==1, SL/TIME => label_A==0
bad_tp = df[(er.isin(["TP_LONG","TP_SHORT"])) & (df["label_A"] != 1)]
bad_non_tp = df[(~er.isin(["TP_LONG","TP_SHORT"])) & (df["label_A"] != 0)]

print("\nBad TP rows (should be 0):", len(bad_tp))
print("Bad non-TP rows (should be 0):", len(bad_non_tp))

assert len(bad_tp) == 0
assert len(bad_non_tp) == 0

filled share (loose, not NOFILL): 0.4531140568064568
filled share (strict, not NOFILL/CONFIRM_FAIL): 0.3673855101513379

exit_reason (filled only):
label_A_exit_reason
TIME            0.599994
CONFIRM_FAIL    0.189199
SL_SHORT         0.10295
SL_LONG         0.099781
TP_SHORT        0.004241
TP_LONG         0.003823
NONE            0.000013
Name: proportion, dtype: Float64

Bad TP rows (should be 0): 0
Bad non-TP rows (should be 0): 0


In [10]:
# --- 6) 
def q(s, ps=(0.01,0.05,0.1,0.25,0.5,0.75,0.9,0.95,0.99)):
    return s.quantile(list(ps))

print("audit_tp_bps quantiles:\n", q(df["audit_tp_bps"]))
print("\naudit_sl_bps quantiles:\n", q(df["audit_sl_bps"]))
print("\naudit_cost_bps quantiles:\n", q(df["audit_cost_bps"]))
print("\naudit_cost_R quantiles:\n", q(df["audit_cost_R"]))
print("\naudit_p_thr_ev0 quantiles:\n", q(df["audit_p_thr_ev0"]))

# corr rapide
corr = df[["audit_cost_R","audit_p_thr_ev0","audit_tp_bps","audit_sl_bps","audit_cost_bps"]].corr(numeric_only=True)
print("\n--- correlations ---")
display(corr)

audit_tp_bps quantiles:
 0.01    24.000000
0.05    24.000000
0.10    24.000000
0.25    24.000000
0.50    24.000000
0.75    28.746250
0.90    39.407494
0.95    44.291847
0.99    68.187607
Name: audit_tp_bps, dtype: float64

audit_sl_bps quantiles:
 0.01     8.000000
0.05     8.000000
0.10     8.000000
0.25     8.000000
0.50     8.000000
0.75    10.519128
0.90    15.657568
0.95    19.901711
0.99    30.986866
Name: audit_sl_bps, dtype: float64

audit_cost_bps quantiles:
 0.01    6.006915
0.05    6.007055
0.10    6.007165
0.25    6.007387
0.50    6.007863
0.75    6.009990
0.90    6.011687
0.95    6.011983
0.99    6.012634
Name: audit_cost_bps, dtype: float64

audit_cost_R quantiles:
 0.01    0.194432
0.05    0.301982
0.10    0.383848
0.25    0.571247
0.50    0.750902
0.75    0.751017
0.90    0.751449
0.95    0.751468
0.99    0.751559
Name: audit_cost_R, dtype: float64

audit_p_thr_ev0 quantiles:
 0.01    0.373260
0.05    0.387193
0.10    0.399662
0.25    0.418103
0.50    0.437727
0.75    0

,audit_cost_R,audit_p_thr_ev0,audit_tp_bps,audit_sl_bps,audit_cost_bps
audit_cost_R,1.000000,0.881860,-0.833150,-0.859496,-0.040092
audit_p_thr_ev0,0.881860,1.000000,-0.820072,-0.795837,-0.038038
audit_tp_bps,-0.833150,-0.820072,1.000000,0.993476,0.106652
audit_sl_bps,-0.859496,-0.795837,0.993476,1.000000,0.099473
audit_cost_bps,-0.040092,-0.038038,0.106652,0.099473,1.000000


In [11]:
# --- 7) 
tp_mask = df["label_A"] == 1
df_tp = df.loc[tp_mask].copy()

print("TP count:", len(df_tp))
if len(df_tp) > 0:
    print("\nTP pnl_net_bps quantiles:\n", df_tp["label_A_pnl_net_bps"].quantile([0.01,0.05,0.1,0.25,0.5,0.75,0.9,0.95,0.99]))

    # outliers (top 10)
    print("\nTop 10 TP pnl:")
    display(df_tp.sort_values("label_A_pnl_net_bps", ascending=False).head(10)[
        ["id_t","label_A_exit_reason","label_A_pnl_net_bps","audit_tp_bps","audit_cost_bps","audit_R_bps","audit_cost_R"]
    ])

# TP counts par mois
tp_month = df_tp["id_t"].dt.to_period("M").astype(str).value_counts().sort_index()
print("\nTP counts per month:")
display(tp_month)

TP count: 1852

TP pnl_net_bps quantiles:
 0.01    17.988258
0.05    17.991830
0.10    17.992492
0.25    20.496589
0.50    25.285886
0.75    35.403691
0.90    44.154680
0.95    52.907193
0.99    91.552557
Name: label_A_pnl_net_bps, dtype: float64

Top 10 TP pnl:


,id_t,label_A_exit_reason,label_A_pnl_net_bps,audit_tp_bps,audit_cost_bps,audit_R_bps,audit_cost_R
287537,2024-04-13 20:09:00+00:00,TP_LONG,253.387329,261.819458,8.432127,119.008842,0.070853
287536,2024-04-13 20:08:30+00:00,TP_SHORT,216.412735,222.518478,6.105739,101.144760,0.060366
287535,2024-04-13 20:08:00+00:00,TP_SHORT,216.180817,222.518478,6.337669,101.144760,0.062659
25608,2024-01-09 21:24:00+00:00,TP_SHORT,173.038757,179.049515,6.010771,81.386147,0.073855
25609,2024-01-09 21:24:30+00:00,TP_SHORT,173.038696,179.049515,6.010821,81.386147,0.073856
25606,2024-01-09 21:23:00+00:00,TP_SHORT,170.594849,176.605682,6.010831,80.275314,0.074878
25607,2024-01-09 21:23:30+00:00,TP_SHORT,169.937836,176.605682,6.667857,80.275314,0.083062
287532,2024-04-13 20:06:30+00:00,TP_SHORT,155.247162,161.727982,6.480829,73.512726,0.088159
287530,2024-04-13 20:05:30+00:00,TP_SHORT,148.767197,154.775024,6.007830,70.352287,0.085396
25593,2024-01-09 21:16:30+00:00,TP_SHORT,145.730148,151.836853,6.106713,69.016754,0.088482



TP counts per month:


/var/folders/my/rlps7sd127bcnxcjs29ybzfm0000gn/T/ipykernel_38375/1384118677.py:16: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  tp_month = df_tp["id_t"].dt.to_period("M").astype(str).value_counts().sort_index()


id_t
2024-01    301
2024-02    183
2024-03    491
2024-04    484
2024-05    257
2024-06    136
Name: count, dtype: int64